# 🦷 Fine-grained Classification เครื่องมือผ่าตัด — DINOv2 + Length Fusion + ArcFace

Pipeline สำหรับจำแนก **เครื่องมือผ่าตัด 14 classes** (~400 ภาพ) โลหะสีเงินบนถาดสีเงิน บางคู่ class ต่างกันแค่ความยาว:

```
ภาพ ──► DINOv2 ViT-S/14 ──► CLS embedding (384-dim)──┐
                                                      ├─ concat(385) ─► Linear ─► 384 ─► ArcFace loss
mask ──► minAreaRect ──► ความยาว (normalize) ─────────┘
```

**สิ่งที่ notebook นี้ทำ:** ติดตั้ง deps → เขียนโมดูลทั้ง 6 ไฟล์ (`config/dataset/model/train/evaluate/infer`) → sanity-check ข้อมูล → เทรน (LoRA + warmup/cosine + early stopping) → confusion matrix → inference demo

**ข้อมูลที่ต้องเตรียม:** Roboflow export แบบ *COCO Segmentation* ที่มีโครงสร้าง
`DATA_DIR/train/_annotations.coco.json` + `DATA_DIR/valid/_annotations.coco.json`


## 0) ติดตั้ง dependencies

In [ ]:
%pip install -q -U torchao peft transformers pytorch-metric-learning albumentations opencv-python-headless scikit-learn seaborn tqdm
print('✅ deps ready')

## 1) เตรียมข้อมูล — เลือก **วิธีใดวิธีหนึ่ง** แล้วรันเซลล์นี้

In [ ]:
# ── กำหนด DATA_DIR ให้ตรงกับที่ข้อมูลอยู่ ──────────────────────────────
DATA_DIR = "/content/dental_dataset"

# ▸ วิธี A: อัปโหลดไฟล์ .zip (export จาก Roboflow แล้ว zip ไว้)
# from google.colab import files
# up = files.upload()                       # เลือก dataset.zip
# !unzip -o -q "{list(up.keys())[0]}" -d /content/
# DATA_DIR = "/content/dataset"             # ← แก้ตามชื่อโฟลเดอร์ข้างใน zip

# ▸ วิธี B: ไฟล์อยู่บน Google Drive อยู่แล้ว
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/path/to/dataset"

# ▸ วิธี C: ดึงตรงจาก Roboflow (แก้ API_KEY / WORKSPACE / PROJECT / VERSION)
# %pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# rf.workspace("WORKSPACE").project("PROJECT").version(VERSION)\
#   .download("coco-segmentation", location=DATA_DIR)

import pathlib
assert pathlib.Path(DATA_DIR, "train", "_annotations.coco.json").exists(), \
    f"ไม่พบ {DATA_DIR}/train/_annotations.coco.json — รันเซลล์ import ข้อมูลก่อน"
for sp in ["train", "valid", "test"]:
    p = pathlib.Path(DATA_DIR, sp)
    if p.exists():
        n_img = len(list(p.glob('*.jpg'))) + len(list(p.glob('*.png'))) + len(list(p.glob('*.jpeg')))
        n_json = len(list(p.glob('*annotations*.json')))
        print(f"{sp:6s}: {n_img:4d} รูป, {n_json} annotation file")

## 2) เขียนโมดูลโค้ดลงไฟล์ (%%writefile)

แต่ละเซลล์สร้างไฟล์ .py ตามหน้าที่ — แก้โค้ดได้จากไฟล์ editor ด้านซ้ายของ Colab หลังจากรันแล้ว

In [ ]:
%%writefile config.py
# -*- coding: utf-8 -*-
"""
config.py — ค่าตั้งต้นกลางของทั้ง pipeline

แก้ค่าในไฟล์นี้ หรือ override ตอนสร้าง object: TrainConfig(data_dir=..., epochs=...)
"""
from dataclasses import asdict, dataclass
from typing import List, Optional


@dataclass
class TrainConfig:
    # ---------------- ข้อมูล ----------------
    data_dir: str = "dataset"          # โฟลเดอร์ที่มี train/ และ valid/ (Roboflow COCO Segmentation export)
    img_size: int = 616                # ต้องหารด้วย 14 ลงตัว (616=44×14) — DINOv2 patch 14
    val_fraction: float = 0.2          # ใช้เมื่อไม่มีโฟลเดอร์ valid/ → stratified split จาก train
    calibration_ratio: Optional[float] = None  # cm/pixel — วัดจาก object อ้างอิงที่รู้ขนาดจริง
                                               # (กล้องระยะคงที่ตลอด จึงใช้ค่าเดียวได้ทุกภาพ)
                                               # None = ใช้หน่วย pixel แล้ว normalize ด้วย mean/std ของ train
    flip_allowed: Optional[List[str]] = None   # รายชื่อ class ที่ "flip ซ้าย-ขวาได้"
                                               # None = flip ได้ทุก class
                                               # class ที่มี handedness (ของซ้าย/ขวามือ) → ตัดชื่อออกจากลิสต์
    num_workers: int = 2               # Colab/Linux ใช้ 2 ได้ — Windows ถ้าค้างให้เปลี่ยนเป็น 0
    bbox_margin: float = 0.15          # crop รอบ bbox ต่อชิ้น (0=ไม่ crop, 0.15 ดีสุด 0.9688)

    # ---------------- โมเดล ----------------
    backbone_name: str = "facebook/dinov2-small"  # ViT-S/14, hidden dim = 384 (~21M params)
    finetune_mode: str = "lora"        # "lora" (แนะนำสำหรับข้อมูลน้อย) | "partial" | "frozen"
    partial_last_blocks: int = 2       # ใช้เมื่อ mode="partial": ปลดล็อก N ViT block ท้าย + final LayerNorm
    lora_r: int = 16                   # ★ ดีสุดจาก tune 0.9688 (r8 ได้ 0.9467) — r16 capacity เยอะขึ้น
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    head_dropout: float = 0.1          # dropout ของ fusion head
    use_attention_pool: bool = True    # True = AttentionPooling แทน CLS-only (v2)
                                      # False = ใช้ CLS token เหมือนเดิม (backward compat)
    mixup_alpha: float = 0.4           # Mixup alpha สำหรับ regularization (0.0 = ปิด)
                                      # ยิ่งน้อยยิ่งแรง, 0.4 เหมาะกับ ~30 samples/class
    use_tta: bool = True               # เปิด TTA ตอน inference (flip + multi-scale)
    # ---------------- SEF (Scharr Edge Fusion) ----------------
    use_sef: bool = False                # เปิด Scharr edge branch (เพิ่ม 64-dim edge feature)

    # ---------------- ArcFace ----------------
    margin: float = 28.6               # additive angular margin หน่วย "องศา" (≈ 0.5 rad)
                                       # — pytorch-metric-learning รับหน่วยองศาแล้วแปลงเป็นเรเดียนเอง
    scale: float = 64.0                # s: ขยาย cosine ก่อน softmax เพื่อไม่ให้ gradient แบน

    # ---------------- CAHM (Confusion-Aware Hard Mining) ----------------
    use_cahm: bool = False
    cahm_alpha: float = 2.0            # น้ำหนักเสริมสำหรับคู่สับสน
    cahm_beta: float = 0.9             # EMA smoothing ของ difficulty score
    cahm_start_epoch: int = 10         # เริ่มใช้หลัง epoch นี้ (ให้ confusion นิ่งก่อน)

    # ---------------- LGMS (Length-Gated Margin Scaling) ----------------
    use_lgms: bool = False
    lgms_gamma: float = 10.0           # องศาเพิ่มสูงสุดสำหรับ margin ของ class ที่ length ใกล้กัน
    lgms_k: int = 2                    # จำนวน twin classes ที่ใกล้ที่สุด

    # ---------------- การเทรน ----------------
    batch_size: int = 32               # เต็มที่ T4 15GB (616px + DINOv2-S + LoRA)
    epochs: int = 50
    patience: int = 12                 # early stopping: stop when val accuracy stops improving for N epochs
    lr_head: float = 3e-4              # learning rate ของ fusion head
    lr_backbone: float = 5e-6          # ใช้เมื่อ finetune_mode="partial"
    lr_lora: float = 1e-4              # ใช้เมื่อ finetune_mode="lora"
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1          # 10% แรกของ total steps เป็น linear warmup แล้วค่อย cosine decay
    grad_clip: float = 1.0
    seed: int = 42
    kfold: Optional[int] = None        # เช่น 5 = Stratified 5-fold CV (แม่นกว่าเมื่อข้อมูลน้อย)
    output_dir: str = "outputs"

    def to_dict(self) -> dict:
        """แปลง config เป็น dict (เก็บลง checkpoint เพื่อ reproduce ตอน evaluate/infer)"""
        return asdict(self)


In [ ]:
%%writefile dataset.py
# -*- coding: utf-8 -*-
"""
dataset.py — โหลดภาพ + segmentation mask จาก COCO format (Roboflow export)

หน้าที่หลัก:
1) parse ``_annotations.coco.json`` → records (path, polygon, label)
2) rasterize polygon → binary mask
3) วัดความยาวเครื่องมือจาก mask (minAreaRect) เป็น auxiliary feature 1 ค่า
4) augmentation ที่ปลอดภัยกับงานนี้:
   - เน้น photometric (brightness/contrast/gamma/CLAHE) จำลองแสงสะท้อนบนโลหะ
   - ไม่มี crop/zoom ที่ทำลาย aspect ratio หรือ scale ของวัตถุ (ขนาดคือฟีเจอร์สำคัญ!)
   - ไม่มี cutout/random erasing กลางวัตถุ
   - horizontal flip เปิด/ปิดได้ "ต่อ class" (บาง class มี handedness ห้าม flip)
"""
import json
import math
import os
import random
from typing import List, Optional, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN: Tuple[float, ...] = (0.485, 0.456, 0.406)
IMAGENET_STD: Tuple[float, ...] = (0.229, 0.224, 0.225)


# ============================================================ การวัดความยาวจาก mask
def measure_length_px(mask: np.ndarray) -> float:
    """
    คืนความยาวสูงสุดของเครื่องมือ หน่วย pixel (จาก binary mask ชิ้นเดียว)

    ใช้ ``cv2.minAreaRect`` เพราะเครื่องมือมักวาง "เฉียง" ไม่ตรงแนวแกน —
    กล่องหมุนที่มี area เล็กสุดที่ล้อม contour จะให้ด้านยาวสุด ≈ ความยาวจริงของเครื่องมือ
    """
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:  # mask ว่าง (annotation ผิดพลาด) → คืน 0 กัน crash
        return 0.0
    largest = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(largest)  # ((cx,cy), (w,h), angle)
    (w, h) = rect[1]
    return float(max(w, h))


def get_length_cm(mask: np.ndarray, calibration_ratio: float) -> float:
    """แปลงความยาว pixel → cm ด้วย calibration_ratio (cm/pixel) จาก object อ้างอิง"""
    return measure_length_px(mask) * calibration_ratio


def mask_from_coco_segmentation(segmentation, height: int, width: int) -> np.ndarray:
    """
    แปลง segmentation ของ COCO → binary mask (uint8, ค่า 0/255)

    รองรับ polygon (รูปแบบมาตรฐานของ Roboflow) และ RLE (ต้องมี pycocotools)
    """
    mask = np.zeros((height, width), dtype=np.uint8)
    if isinstance(segmentation, dict):  # RLE format
        try:
            from pycocotools import mask as mask_utils
        except ImportError as exc:
            raise ImportError(
                "เจอ segmentation แบบ RLE แต่ยังไม่ได้ติดตั้ง pycocotools (pip install pycocotools)"
            ) from exc
        rle = segmentation
        if isinstance(rle.get("counts"), list):  # uncompressed RLE → แปลงเป็น compressed ก่อน
            rle = mask_utils.frPyObjects(rle, height, width)
        return (mask_utils.decode(rle) * 255).astype(np.uint8)
    for poly in segmentation:  # list ของ polygon [[x1,y1,x2,y2,...], ...]
        pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
        cv2.fillPoly(mask, [np.round(pts).astype(np.int32)], 255)
    return mask


# ============================================================ COCO parsing
def load_coco_records(data_dir: str, split: str) -> Tuple[List[dict], List[str]]:
    """
    อ่านโฟลเดอร์ split ("train"/"valid"/"test") ที่มี ``_annotations.coco.json``

    คืน ``(records, class_names)`` โดย record = dict ที่มี
    ``image_path / segmentation / width / height / class_name / label``

    - label = index จากการ **sort ชื่อ class** (คงที่เสมอ ไม่ว่า category_id ใน json จะเรียงแบบไหน)
    - 1 annotation = 1 sample → ถ้า 1 ภาพมีหลายเครื่องมือ จะได้หลาย sample
      (แนะนำใช้ bbox_margin > 0 ใน Dataset เพื่อ crop รายชิ้น)
    """
    split_dir = os.path.join(data_dir, split)
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    if not os.path.exists(ann_path):
        raise FileNotFoundError(f"ไม่เจอไฟล์ annotation: {ann_path}")
    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = {im["id"]: im for im in coco["images"]}
    cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    # Filter to categories that actually appear (skip dummy super-category like id 0)
    used_names = {cat_id_to_name[ann["category_id"]] for ann in coco["annotations"]}
    class_names = sorted(used_names)
    name_to_label = {n: i for i, n in enumerate(class_names)}

    records: List[dict] = []
    for ann in coco["annotations"]:
        im = images[ann["image_id"]]
        cname = cat_id_to_name[ann["category_id"]]
        records.append({
            "image_path": os.path.join(split_dir, im["file_name"]),
            "segmentation": ann["segmentation"],
            "width": int(im["width"]),
            "height": int(im["height"]),
            "class_name": cname,
            "label": name_to_label[cname],
        })
    return records, class_names


def segmentation_bbox(segmentation, width: int, height: int) -> Tuple[int, int, int, int]:
    """bounding box (x1,y1,x2,y2) ครอบ polygon ทั้งหมด — ใช้ตอน crop รายชิ้น (bbox_margin > 0)"""
    xs: List[float] = []
    ys: List[float] = []
    for poly in segmentation if isinstance(segmentation, list) else []:
        pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
        xs += [float(pts[:, 0].min()), float(pts[:, 0].max())]
        ys += [float(pts[:, 1].min()), float(pts[:, 1].max())]
    if not xs:  # RLE หรือ polygon ว่าง → ใช้ทั้งภาพ
        return 0, 0, width, height
    return int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))


# ============================================================ Augmentation
def build_photometric_aug() -> A.Compose:
    """
    Augmentation ฝั่ง "แสง" สำหรับ train — ตั้งใจให้แรงกว่าปกติ เพราะโลหะสะท้อนแสง
    ต่างกันทุกครั้งที่ถ่าย แต่ *ไม่มี* การ crop/scale/cutout ใดๆ เพราะ
    "ขนาดและรูปทรง" คือสิ่งที่โมเดลต้องเรียนรู้เพื่อแยก class ที่เหมือนกัน
    """
    return A.Compose([
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),             # เพิ่ม local contrast (โลหะบนถาดโลหะ contrast ต่ำ)
        A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.7),  # jitter แรงกว่าปกติ
        A.RandomGamma(gamma_limit=(70, 150), p=0.7),                        # จำลอง exposure/แสงไฟต่างกัน
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=15, p=0.3),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),                           # เบลอเล็กน้อยแบบ defocus
    ])


def build_tensor_transform(img_size: int) -> A.Compose:
    """resize ขนาดคงที่ + normalize ด้วยค่า ImageNet + แปลงเป็น tensor (ใช้ทั้ง train/eval/infer)"""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def scharr_edge_map(gray: np.ndarray) -> np.ndarray:
    """
    คำนวณ Scharr edge magnitude จากภาพ grayscale (uint8) — คืน float32 [0,1]
    ใช้สำหรับ SEF branch: เน้นขอบ/รูปทรงของเครื่องมือให้ชัดกว่าสี
    """
    gx = cv2.Scharr(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Scharr(gray, cv2.CV_32F, 0, 1)
    mag = cv2.magnitude(gx, gy)
    m = float(mag.max())
    if m > 1e-6:
        mag = mag / m
    return mag.astype(np.float32)


def compute_class_length_means(records: List[dict], calibration_ratio: Optional[float] = None) -> dict:
    """เฉลี่ยความยาวต่อ class (สำหรับ LGMS) — ใช้ train records เท่านั้น"""
    from collections import defaultdict
    sums: dict = defaultdict(list)
    for r in records:
        mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        L = measure_length_px(mask)
        if calibration_ratio is not None:
            L *= calibration_ratio
        sums[r["label"]].append(L)
    return {k: float(np.mean(v)) for k, v in sums.items()}


def compute_lgms_margins(class_means: dict, num_classes: int, m_base: float = 28.6,
                         gamma: float = 10.0, k: int = 2) -> List[float]:
    """
    คำนวณ margin ต่อ class สำหรับ LGMS
    sim_len(i,j) = 1 - |len_i - len_j| / max_diff
    m(y) = m_base + gamma * mean(sim ของ k คู่ใกล้สุด)
    """
    means = np.array([class_means.get(i, 0.0) for i in range(num_classes)], dtype=np.float64)
    if num_classes <= 1:
        return [m_base] * num_classes
    diff = np.abs(means[:, None] - means[None, :])  # (C,C)
    max_diff = float(diff.max())
    if max_diff < 1e-6:
        return [m_base] * num_classes
    sim = 1.0 - diff / max_diff  # 0..1, สูง = ใกล้กัน
    np.fill_diagonal(sim, -1)  # ไม่นับตัวเอง
    margins = []
    for y in range(num_classes):
        # k อันดับสูงสุด
        topk_idx = np.argsort(sim[y])[::-1][:k]
        # กรองค่าลบ (กรณี k > C-1)
        vals = [sim[y, j] for j in topk_idx if sim[y, j] >= 0]
        mean_sim = float(np.mean(vals)) if vals else 0.0
        margins.append(float(m_base + gamma * mean_sim))
    return margins

def simulate_shadow(img: np.ndarray, rng: Optional[random.Random] = None) -> np.ndarray:
    """
    จำลอง "เงา" บนพื้นหลังสีเขียว (green mat) — เงาเคลื่อนตามตำแหน่งวางเครื่องมือ/ทิศไฟ

    วาด blob มืดขอบนุ่ม 1-2 จุด (ellipse + Gaussian falloff) คูณลงภาพ
    เขียน numpy เองแทน A.RandomShadow เพราะ signature ของ albumentations
    เปลี่ยนบ่อยระหว่าง 1.x ↔ 2.x — ไม่อยากผูกกับเวอร์ชัน
    """
    rng = rng or random
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    mask = np.ones((h, w), dtype=np.float32)
    for _ in range(rng.randint(1, 3)):
        cx, cy = rng.uniform(0, w), rng.uniform(0, h)
        ax = rng.uniform(w * 0.2, w * 0.7)
        ay = rng.uniform(h * 0.2, h * 0.7)
        strength = rng.uniform(0.35, 0.65)          # เงาดำสุด ~35-65%
        d2 = ((xx - cx) / ax) ** 2 + ((yy - cy) / ay) ** 2
        mask *= 1.0 - strength * np.exp(-d2)
    out = img.astype(np.float32) * mask[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)


# ============================================================ สถิติความยาว + split
def record_lengths(records: List[dict], calibration_ratio: Optional[float]) -> List[float]:
    """วัดความยาวของทุก record (px หรือ cm ถ้ามี ratio) — rasterize จาก polygon โดยตรง"""
    out = []
    for r in records:
        mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        L = measure_length_px(mask)
        out.append(L if calibration_ratio is None else L * calibration_ratio)
    return out


def compute_length_stats(records: List[dict], calibration_ratio: Optional[float] = None) -> Tuple[float, float]:
    """
    คำนวณ mean/std ของความยาว — **ต้องคำนวณจาก train เท่านั้น**
    แล้วใช้ค่าเดียวกันนี้ normalize ทั้ง val/test/inference (กัน data leakage)
    """
    L = np.asarray(record_lengths(records, calibration_ratio), dtype=np.float64)
    mean = float(L.mean())
    std = max(float(L.std()), 1e-6)
    return mean, std


def stratified_split(records: List[dict], val_fraction: float = 0.2, seed: int = 42):
    """แบ่ง train/val แบบ stratified (คงสัดส่วน class) — จำเป็นเมื่อข้อมูลน้อย ~30 ภาพ/class"""
    if val_fraction <= 0 or len(records) < 10:
        return records, []
    from sklearn.model_selection import train_test_split
    y = [r["label"] for r in records]
    tr, va = train_test_split(records, test_size=val_fraction, random_state=seed, stratify=y)
    return tr, va


# ============================================================ PyTorch Dataset
class SurgicalInstrumentDataset(Dataset):
    """
    Dataset สำหรับงานจำแนกเครื่องมือผ่าตัด — คืน dict:
      ``image``  : FloatTensor (3, H, W) normalize แล้ว
      ``length`` : scalar float = (ความยาว − mean) / std  ← auxiliary feature
      ``label``  : int64 class index

    หมายเหตุสำคัญ:
    - ความยาววัดจาก mask "ต้นฉบับ" (ก่อน augment) เพราะ photometric/flip ไม่ควรเปลี่ยนความยาวจริง
    - ``flip_flags[label]`` เป็น True เท่านั้นที่จะโดน horizontal flip
    - ``bbox_margin`` > 0 เมื่อ 1 ภาพมีหลายเครื่องมือ → crop รอบ bbox ของชิ้นนั้น
      (crop คงสัดส่วน/scale เดิม ไม่ใช่การ zoom อิสระ)
    """

    def __init__(self, records: List[dict], length_stats: Tuple[float, float],
                 img_size: int = 224, calibration_ratio: Optional[float] = None,
                 flip_flags: Optional[List[bool]] = None, training: bool = True,
                 bbox_margin: float = 0.0, use_sef: bool = False):
        self.records = records
        self.length_mean, self.length_std = length_stats
        self.training = training
        self.bbox_margin = bbox_margin
        self.img_size = img_size
        self.use_sef = use_sef
        self.tensor_tf = build_tensor_transform(img_size)
        self.aug = build_photometric_aug() if training else None
        self.flip_flags = flip_flags
        # วัดความยาวครั้งเดียวตอนสร้าง dataset (rasterize polygon ใน memory เร็วมาก)
        self._lengths = record_lengths(records, calibration_ratio)

    def __len__(self) -> int:
        return len(self.records)

    def _maybe_crop(self, img: np.ndarray, r: dict) -> np.ndarray:
        m = self.bbox_margin
        x1, y1, x2, y2 = segmentation_bbox(r["segmentation"], r["width"], r["height"])
        dx, dy = int((x2 - x1) * m), int((y2 - y1) * m)
        x1, y1 = max(x1 - dx, 0), max(y1 - dy, 0)
        x2, y2 = min(x2 + dx, r["width"]), min(y2 + dy, r["height"])
        return img[y1:y2, x1:x2]

    def __getitem__(self, idx: int) -> dict:
        r = self.records[idx]
        img = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if img is None:
            raise IOError(f"อ่านภาพไม่สำเร็จ: {r['image_path']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.bbox_margin > 0:
            img = self._maybe_crop(img, r)
        if self.training:
            if random.random() < 0.5:               # เงาพื้นเขียว (โจทย์จริง: แสง/เงาเป็นปัญหาหลัก)
                img = simulate_shadow(img)
            img = self.aug(image=img)["image"]
            # horizontal flip แบบ per-class (เฉพาะ class ที่ไม่มีปัญหา handedness)
            if self.flip_flags is not None and self.flip_flags[r["label"]] and random.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1, :])

        # tensor หลัก
        tensor = self.tensor_tf(image=img)["image"]
        length_norm = (self._lengths[idx] - self.length_mean) / self.length_std
        out = {
            "image": tensor,
            "length": torch.tensor(length_norm, dtype=torch.float32),
            "label": torch.tensor(r["label"], dtype=torch.long),
        }
        # SEF: คำนวณ Scharr edge map จากภาพหลัง augment/flip แล้ว resize เป็น img_size
        if self.use_sef:
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            edge = scharr_edge_map(gray)  # (H,W) float [0,1]
            edge_resized = cv2.resize(edge, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
            out["edge_map"] = torch.from_numpy(edge_resized).unsqueeze(0).float()  # (1,H,W)
        return out


# ============================================================ Visualization (debug)
def visualize_records(records: List[dict], calibration_ratio: Optional[float] = None,
                      n: int = 6, cols: int = 3, seed: int = 0):
    """
    แสดงภาพ + เส้น outline ของ mask + ความยาวที่วัดได้ — ใช้เช็คว่า
    annotation/การวัดความยาวถูกต้อง ก่อนเทรนจริง (คืน matplotlib figure)
    """
    import matplotlib.pyplot as plt
    rng = random.Random(seed)
    picks = rng.sample(records, min(n, len(records)))
    rows = math.ceil(len(picks) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.6 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(picks):]:
        ax.axis("off")
    unit = "cm" if calibration_ratio else "px"
    for ax, r in zip(axes, picks):
        bgr = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, cnts, -1, (255, 40, 40), 3)
        L = measure_length_px(mask) * (calibration_ratio if calibration_ratio else 1.0)
        ax.imshow(img)
        ax.set_title(f"{r['class_name']} | {L:.1f} {unit}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    return fig


In [ ]:
%%writefile model.py
# -*- coding: utf-8 -*-
"""
model.py — DINOv2 backbone + fusion head ที่รวม "ความยาวจาก mask" เข้ากับ embedding

สถาปัตยกรรม (v2 — ปรับปรุง):
    ภาพ → DINOv2 ViT-S/14 → all tokens (257 tokens, 384-dim)
    → AttentionPooling: multi-head attention เฉพาะ patch tokens → weighted sum → 384-dim
    mask → measure_length_px() → normalize(mean,std ของ train) → scalar (1-dim)
    concat(384+1) → DeepFusionHead (LN→Linear→GELU→Dropout→Linear→GELU→Dropout) → 384-dim
    → ArcFace loss (normalize L2 ทั้ง embedding และ weight class)

v2 changes:
  - AttentionPooling แทน CLS-only: เก็บ spatial detail จาก patch tokens ทั้งหมด
  - DeepFusionHead: 2-layer MLP with LayerNorm สำหรับ fusion ที่ลึกกว่า
  - รองรับ mask_aux_features: width/height/area/ratio จาก mask (optional, เพิ่ม auxiliary info)

เหตุผลที่ต้อง fuse ความยาว: resize ภาพเป็น 224×224 ทำให้ข้อมูล "scale จริง" หายไป
แต่บางคู่ class ต่างกันแค่ควายาว — จึงป้อนความยาวที่วัดจาก mask ตรง ๆ เข้าไปช่วย
"""
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import Dinov2Model

try:
    from edge_branch import ScharrEdgeBranch
    _EDGE_AVAILABLE = True
except ImportError:
    ScharrEdgeBranch = None  # type: ignore
    _EDGE_AVAILABLE = False

try:
    from peft import LoraConfig, TaskType, get_peft_model
    _PEFT_AVAILABLE = True
    _PEFT_IMPORT_ERROR = ""
except ImportError as e:
    # Colab has torchao 0.10.0 but peft 0.17+ requires >=0.16.0
    # Fall back to non-LoRA modes; user can fix via: !pip install -U torchao  (then restart runtime)
    _PEFT_AVAILABLE = False
    _PEFT_IMPORT_ERROR = str(e)
except Exception as e:
    _PEFT_AVAILABLE = False
    _PEFT_IMPORT_ERROR = str(e)


class AttentionPooling(nn.Module):
    """
    Multi-head attention pooling บน patch tokens ของ ViT

    แทนที่จะใช้ CLS token อย่างเดียว — เลือก weight จาก attention
    เฉพาะ patch tokens (ไม่เอา CLS) เพื่อเก็บ spatial detail
    ที่จำเป็นสำหรับ fine-grained classification
    """

    def __init__(self, embed_dim: int, num_heads: int = 6, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(embed_dim)
        # learnable query — 1 token ที่ attend เข้า patch tokens ทั้งหมด
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, num_tokens, embed_dim) — tokens จาก ViT (รวม CLS)
        return: (B, embed_dim) — pooled embedding
        """
        B = x.shape[0]
        # แยก patch tokens (index 1:) ออกจาก CLS (index 0)
        patch_tokens = x[:, 1:, :]   # (B, 256, 384)
        q = self.query.expand(B, -1, -1)  # (B, 1, 384)
        attn_out, _ = self.attn(q, patch_tokens, patch_tokens)  # (B, 1, 384)
        attn_out = attn_out.squeeze(1)   # (B, 384)
        return self.norm(attn_out)


class DeepFusionHead(nn.Module):
    """
    Deep fusion head: concat visual embedding + length scalar → project กลับ 384-dim

    v1: Linear(385→384) ชั้นเดียว — เร็วแต่ fusion ตื้น
    v2: LN → Linear(385→768) → GELU → Dropout → Linear(768→384) → Dropout
    """

    def __init__(self, embed_dim: int, aux_dim: int = 1, dropout: float = 0.1):
        super().__init__()
        self.ln = nn.LayerNorm(embed_dim + aux_dim)
        self.net = nn.Sequential(
            nn.Linear(embed_dim + aux_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, emb: torch.Tensor, aux: torch.Tensor) -> torch.Tensor:
        """
        emb: (B, embed_dim) | aux: (B, aux_dim)
        return: (B, embed_dim)
        """
        x = torch.cat([emb, aux.to(emb.dtype)], dim=1)  # (B, embed_dim+aux_dim)
        x = self.ln(x)
        return self.net(x)


class SurgicalDinoFusion(nn.Module):
    """
    DINOv2 backbone (+LoRA/ปลดล็อกบางส่วน) + AttentionPooling + DeepFusionHead

    finetune_mode:
      - "lora":    หลอด backbone ด้วย LoRA (train ~เฉพาะ adapter, พารามิเตอร์น้อยมาก)
      - "partial": freeze ทั้ง backbone แล้วปลดล็อก N block ท้าย + final LayerNorm
      - "frozen":  freeze backbone ทั้งหมด (ใช้เป็น feature extractor อย่างเดียว)
    """

    def __init__(self,
                 backbone_name: str = "facebook/dinov2-small",
                 finetune_mode: str = "lora",
                 lora_r: int = 8,
                 lora_alpha: int = 16,
                 lora_dropout: float = 0.1,
                 partial_last_blocks: int = 2,
                 head_dropout: float = 0.1,
                 use_attention_pool: bool = True,
                 use_sef: bool = False,
                 sef_out_dim: int = 64):
        super().__init__()
        assert finetune_mode in ("frozen", "partial", "lora"), f"mode ไม่ถูกต้อง: {finetune_mode}"

        self.backbone = Dinov2Model.from_pretrained(backbone_name)
        self.embed_dim = self.backbone.config.hidden_size  # 384 สำหรับ dinov2-small
        self.use_attention_pool = use_attention_pool
        self.use_sef = use_sef

        if finetune_mode == "lora":
            if not _PEFT_AVAILABLE:
                hint = f" (detail: {_PEFT_IMPORT_ERROR[:120]})" if _PEFT_IMPORT_ERROR else ""
                raise ImportError(
                    "finetune_mode='lora' requires `peft` but import failed" + hint +
                    ". Fix in Colab: !pip install -U torchao  then Runtime -> Restart session, "
                    "or use finetune_mode='frozen'/'partial' to avoid LoRA."
                )
            lora_cfg = LoraConfig(
                task_type=TaskType.FEATURE_EXTRACTION,
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                target_modules=["query", "value"],  # LoRA เฉพาะ projection ของ attention
                bias="none",
            )
            self.backbone = get_peft_model(self.backbone, lora_cfg)
        elif finetune_mode == "partial":
            self._freeze_partial(partial_last_blocks)
        else:  # frozen
            for p in self.backbone.parameters():
                p.requires_grad = False

        # Attention pooling: aggregate patch tokens → 384-dim
        self.attn_pool = AttentionPooling(self.embed_dim, num_heads=6, dropout=head_dropout) if use_attention_pool else None
        # SEF branch (ถ้าเปิด)
        self.edge_branch = None
        if use_sef:
            if not _EDGE_AVAILABLE:
                raise ImportError("use_sef=True requires edge_branch.py")
            self.edge_branch = ScharrEdgeBranch(out_dim=sef_out_dim, dropout=head_dropout)
        # Deep fusion head: concat(384 + 1 + 64_if_sef) → 384
        aux_dim = 1 + (sef_out_dim if use_sef else 0)
        self.fusion = DeepFusionHead(self.embed_dim, aux_dim=aux_dim, dropout=head_dropout)

    def _freeze_partial(self, k: int) -> None:
        """freeze ทั้ง backbone แล้วปลดล็อกเฉพาะ k block ท้าย + final layernorm"""
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.encoder.layer[-k:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.layernorm.parameters():
            p.requires_grad = True

    def forward(self, pixel_values: torch.Tensor, length_feat: torch.Tensor,
                edge_map: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        pixel_values: (B,3,H,W) normalize แล้ว | length_feat: (B,) normalized length
        edge_map: (B,1,H,W) Scharr edge [0,1] ถ้า use_sef=True — ถ้า None จะใช้ศูนย์แทน
        return: embedding (B, embed_dim=384) สำหรับเข้า ArcFace loss
        """
        out = self.backbone(pixel_values=pixel_values).last_hidden_state  # (B, tokens, 384)
        # Attention pooling: attend เฉพาะ patch tokens แทน CLS-only
        if self.attn_pool is not None:
            e = self.attn_pool(out)        # (B, 384)
        else:
            e = out[:, 0]                   # CLS token fallback
        # Fusion: concat visual embedding + length scalar (+ edge 64 ถ้า SEF)
        if self.use_sef and self.edge_branch is not None and edge_map is not None:
            ef = self.edge_branch(edge_map)  # (B, 64)
            aux = torch.cat([length_feat.unsqueeze(1), ef], dim=1)  # (B, 65)
        else:
            aux = length_feat.unsqueeze(1)      # (B, 1)
        return self.fusion(e, aux)          # (B, 384)

    def param_groups(self, lr_head: float, lr_backbone: Optional[float] = None) -> List[dict]:
        """
        แยกกลุ่มพารามิเตอร์เพื่อใช้ learning rate ต่างกัน:
          - head (attention_pool + fusion + edge_branch): lr_head
          - backbone ที่ requires_grad=True (LoRA adapter หรือ block ที่ปลดล็อก): lr_backbone
        """
        head_params = []
        if self.attn_pool is not None:
            head_params += list(self.attn_pool.parameters())
        head_params += list(self.fusion.parameters())
        if self.edge_branch is not None:
            head_params += list(self.edge_branch.parameters())
        groups = [{"params": [p for p in head_params if p.requires_grad], "lr": lr_head}]
        bb_trainable = [p for p in self.backbone.parameters() if p.requires_grad]
        if bb_trainable:
            groups.append({"params": bb_trainable, "lr": lr_backbone if lr_backbone is not None else lr_head})
        return groups

def arcface_logits(loss_fn, embeddings: torch.Tensor) -> torch.Tensor:
    """
    logits ตอน evaluate/inference: ``s · cos(θ)`` (ไม่มี margin — margin ใช้ตอน train เท่านั้น)

    ใช้ ``loss_fn.get_cosine()`` ของ pytorch-metric-learning โดยตรง — CosineSimilarity
    ของ lib normalize ทั้ง embedding และ weight (W เก็บ shape (emb_dim, num_classes))
    ให้เอง จึงถูกต้องบน "มุม" ทุกเวอร์ชันของ lib
    """
    cos = loss_fn.get_cosine(embeddings)  # (B, num_classes), cos ของมุมระหว่างเวกเตอร์
    return cos * loss_fn.scale


class AdaptiveArcFaceLoss(torch.nn.Module):
    """
    ArcFace แบบ per-class margin (สำหรับ LGMS)

    margin_per_class: list/array ขนาด num_classes หน่วยองศา
    แต่ละ sample ใช้ margin ของ label ตัวเอง
    """
    def __init__(self, num_classes: int, embedding_size: int,
                 margin_per_class: List[float], scale: float = 64.0):
        super().__init__()
        from pytorch_metric_learning.distances import CosineSimilarity
        from pytorch_metric_learning.utils import common_functions as c_f
        self.num_classes = num_classes
        self.embedding_size = embedding_size
        self.scale = scale
        self.margin_per_class = np.array(margin_per_class, dtype=np.float64)
        self.margins_rad = np.radians(self.margin_per_class)
        self.W = torch.nn.Parameter(torch.Tensor(embedding_size, num_classes))
        # Xavier init เหมือน pytorch-metric-learning
        torch.nn.init.xavier_uniform_(self.W)
        self.cross_entropy = torch.nn.CrossEntropyLoss(reduction="none")
        self._cosine = CosineSimilarity()
        self._c_f = c_f

    def get_cosine(self, embeddings: torch.Tensor) -> torch.Tensor:
        # Follows pytorch-metric-learning: normalize embeddings and W
        emb_norm = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        W_norm = torch.nn.functional.normalize(self.W, p=2, dim=0)
        return torch.mm(emb_norm, W_norm)

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """
        คืน loss เฉลี่ย (mean) — เรียก compute_loss แบบ per-sample แล้ว mean เอง
        เพื่อให้ CAHM นำไป weight ต่อได้ (ถ้าต้องการ per-sample loss ให้เรียก compute_loss_dict)
        """
        d = self.compute_loss_dict(embeddings, labels)
        return d["losses"].mean()

    def compute_loss_dict(self, embeddings: torch.Tensor, labels: torch.Tensor) -> dict:
        """
        คืน dict {"losses": (B,) per-sample, "logits": (B,C)}
        สำหรับ CAHM ที่ต้อง weight ราย sample
        """
        dtype, device = embeddings.dtype, embeddings.device
        # ย้าย W/margins ให้ตรง device/dtype
        W = self._c_f.to_device(self.W, device=device, dtype=dtype)
        margins = torch.as_tensor(self.margins_rad, device=device, dtype=dtype)  # (C,)
        # cosine (B, C)
        cosine = self.get_cosine(embeddings)  # internal uses self.W — need to ensure uses moved W; get_cosine uses normalized W directly
        # แต่ get_cosine ดู self.W เอง — override ให้ใช้ W ที่ย้ายแล้วไม่ได้; ทำ manual:
        emb_norm = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        W_norm = torch.nn.functional.normalize(W, p=2, dim=0)
        cosine = torch.mm(emb_norm, W_norm)
        # mask
        B = labels.size(0)
        mask = torch.zeros(B, self.num_classes, dtype=dtype, device=device)
        mask[torch.arange(B, device=device), labels] = 1
        cosine_target = cosine[mask == 1]  # (B,)
        angles = torch.acos(torch.clamp(cosine_target, -1 + 1e-7, 1 - 1e-7))
        m_per_sample = margins[labels]  # (B,)
        # cos(theta + m) แบบ ArcFace
        cos_theta_plus_m = torch.cos(angles + m_per_sample)
        cos_theta = torch.cos(angles)
        # keep monotonically decreasing (เช่นเดียวกับ ArcFace)
        # ถ้า theta + m > pi ให้ fallback
        cond = angles <= (np.pi - m_per_sample)
        # m ใน radian ต้องแปลงเป็น tensor สำหรับ sin
        modified = torch.where(cond, cos_theta_plus_m, cos_theta - m_per_sample * torch.sin(torch.as_tensor(m_per_sample)))
        diff = (modified - cosine_target).unsqueeze(1)  # (B,1)
        logits = cosine + (mask * diff)
        logits = logits * self.scale
        losses = self.cross_entropy(logits, labels)  # (B,)
        return {"losses": losses, "logits": logits, "cosine": cosine}

    # ให้เหมือน pytorch-metric-learning: มี attribute W และ scale สำหรับ arcface_logits
    @property
    def W_t(self):
        return self.W.t()


def count_trainable(model: nn.Module) -> int:
    """นับจำนวนพารามิเตอร์ที่ถูกเทรน (ใช้ยืนยันว่า LoRA/frozen ทำงานถูกต้อง)"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile train.py
# -*- coding: utf-8 -*-
"""
train.py — training loop สำหรับ DINOv2 + length fusion + ArcFace

ใช้จาก notebook/สคริปต์:
    from config import TrainConfig
    from train import run_training
    best_ckpt = run_training(TrainConfig(data_dir="/content/dataset"))

หรือ CLI:
    python train.py --data_dir dataset --epochs 50 --finetune_mode lora
    python train.py --data_dir dataset --kfold 5     # Stratified k-fold CV
"""
import argparse
import math
import os
import random
from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

from pytorch_metric_learning.losses import ArcFaceLoss

from config import TrainConfig
from dataset import (SurgicalInstrumentDataset, compute_class_length_means,
                     compute_length_stats, compute_lgms_margins,
                     load_coco_records, stratified_split)
from model import AdaptiveArcFaceLoss, SurgicalDinoFusion, arcface_logits, count_trainable


# ============================================================ utils
def seed_everything(seed: int) -> None:
    """fix seed ของทุก RNG เพื่อ reproduce ผล (สำคัญเมื่อข้อมูลน้อย split เปลี่ยน = ผลเปลี่ยน)"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.4):
    """
    Mixup: สุ่ม interpolation ระหว่าง sample คู่สุ่ม
    x: image tensor (B,3,H,W) | y: label (B,)
    return: mixed_x, y_a, y_b, lam (lambda = interpolation ratio)

    สำคัญสำหรับข้อมูลน้อย: ช่วย regularization โดย "เบลอ" ระหว่าง class
    alpha=0.4 → lam ~ Beta(0.4, 0.4) มักจะอยู่ใกล้ 0 หรือ 1 (ไม่กลาง太多)
    """
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1.0 - lam)  # ให้ lam >= 0.5 เสมอ เพื่อไม่ให้ label สลับกัน
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def torch_load_compat(path: str) -> dict:
    """torch.load ที่รองรับทั้ง torch เก่า/ใหม่ (default weights_only เปลี่ยนใน torch 2.6)"""
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def build_flip_flags(class_names: List[str], cfg: TrainConfig) -> List[bool]:
    """flip ได้เฉพาะ class ที่อยู่ใน cfg.flip_allowed (None = flip ได้ทุก class)"""
    allowed = set(cfg.flip_allowed) if cfg.flip_allowed is not None else None
    return [True if allowed is None else n in allowed for n in class_names]


def resolve_records(cfg: TrainConfig) -> Tuple[List[dict], List[dict], List[str]]:
    """
    อ่าน records จาก data_dir:
      - ถ้ามี valid/_annotations.coco.json → ใช้เลย
      - ถ้าไม่มี → stratified split จาก train ด้วย val_fraction/seed ของ config
    """
    tr, classes = load_coco_records(cfg.data_dir, "train")
    valid_ann = os.path.join(cfg.data_dir, "valid", "_annotations.coco.json")
    if os.path.exists(valid_ann):
        va, classes_valid = load_coco_records(cfg.data_dir, "valid")
        if classes_valid != classes:
            raise ValueError(f"รายชื่อ class ต่างกันระหว่าง train/valid:\n{classes}\n{classes_valid}")
    else:
        tr, va = stratified_split(tr, cfg.val_fraction, cfg.seed)
        print(f"[data] ไม่มีโฟลเดอร์ valid/ → stratified split {len(tr)}/{len(va)} (seed={cfg.seed})")
    return tr, va, classes

def warmup_cosine_factor(step: int, warmup: int, total: int) -> float:
    """LR schedule: linear warmup → cosine decay ลงจนเกือบ 0 ตอนจบ"""
    if step < warmup:
        return step / max(1, warmup)
    t = min((step - warmup) / max(1, total - warmup), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * t))


# ============================================================ CAHM helpers
def _cahm_d_from_cm(cm: np.ndarray) -> np.ndarray:
    """
    คำนวณ pair difficulty จาก confusion matrix:
      d(i,j) = C[i,j] + C[j,i]  (i!=j), normalize ด้วย max
    """
    C = cm.astype(np.float64)
    n = C.shape[0]
    d = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(n):
            if i != j:
                d[i, j] = C[i, j] + C[j, i]
    m = d.max()
    if m > 1e-9:
        d = d / m
    return d


def _cahm_weights(labels: torch.Tensor, d_t: np.ndarray, alpha: float, device) -> torch.Tensor:
    """w = 1 + alpha * max_j d_t[y,j]  (per-sample)"""
    if d_t is None:
        return torch.ones_like(labels, dtype=torch.float32)
    d = torch.as_tensor(d_t, device=device, dtype=torch.float32)  # (C,C)
    # max ต่อแถว (ไม่นับ diagonal ที่เป็น 0 อยู่แล้ว)
    row_max = d.max(dim=1).values  # (C,)
    w = 1.0 + alpha * row_max[labels]
    return w


@torch.no_grad()
def _eval_confusion(model, loss_fn, loader, device, num_classes: int) -> np.ndarray:
    """รันทั้ง validation set เพื่อสร้าง confusion matrix สำหรับ CAHM"""
    from sklearn.metrics import confusion_matrix
    model.eval()
    ys_true, ys_pred = [], []
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        edge = batch.get("edge_map")
        if edge is not None:
            edge = edge.to(device, non_blocking=True)
        emb = model(px, ln, edge)
        logits = arcface_logits(loss_fn, emb.float())
        pred = logits.argmax(dim=1).cpu().numpy()
        ys_true.extend(y.cpu().numpy().tolist())
        ys_pred.extend(pred.tolist())
    cm = confusion_matrix(ys_true, ys_pred, labels=list(range(num_classes)))
    return cm
# ============================================================ epochs
def train_one_epoch(model, loss_fn, loader, optimizer, scheduler, scaler, device, cfg,
                    cahm_d: Optional[np.ndarray] = None) -> float:
    """เทรน 1 epoch → คืนค่า loss เฉลี่ย (ArcFace บน embedding จาก fusion head)"""
    model.train()
    total, seen = 0.0, 0
    mixup_alpha = getattr(cfg, "mixup_alpha", 0.0)
    use_cahm = bool(getattr(cfg, "use_cahm", False)) and cahm_d is not None
    cahm_alpha = float(getattr(cfg, "cahm_alpha", 2.0))
    is_adaptive = hasattr(loss_fn, "compute_loss_dict")
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        edge = batch.get("edge_map")
        if edge is not None:
            edge = edge.to(device, non_blocking=True)

        # Mixup: สุ่ม interpolate ข้อมูล 2 ตัวอย่าง (regularization สำหรับข้อมูลน้อย)
        use_mixup = mixup_alpha > 0 and model.training and not use_cahm  # ปิด mixup เมื่อใช้ CAHM เพื่อให้ weight ชัดเจน
        if use_mixup:
            px, y_a, y_b, lam = mixup_data(px, y, mixup_alpha)
        else:
            y_a, y_b, lam = y, y, 1.0

        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:  # GPU → mixed precision
            with torch.autocast("cuda"):
                emb = model(px, ln, edge)
                if use_mixup:
                    loss = lam * loss_fn(emb.float(), y_a) + (1 - lam) * loss_fn(emb.float(), y_b)
                elif use_cahm:
                    # CAHM weighted loss — ดึง per-sample loss แล้วคูณ w
                    if is_adaptive:
                        d = loss_fn.compute_loss_dict(emb.float(), y)
                        per = d["losses"]  # (B,)
                    else:
                        # ต้องส่ง ref_emb = embeddings เพื่อผ่าน identity check ของ PML
                        ef = emb.float()
                        ld = loss_fn.compute_loss(ef, y, None, ef, y)
                        per = ld["loss"]["losses"]  # (B,)
                    w = _cahm_weights(y, cahm_d, cahm_alpha, device)
                    loss = (per * w).mean()
                else:
                    loss = loss_fn(emb.float(), y)  # cast fp32 ก่อนเข้า ArcFace เพื่อ stability
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:  # CPU → fp32
            emb = model(px, ln, edge)
            if use_mixup:
                loss = lam * loss_fn(emb, y_a) + (1 - lam) * loss_fn(emb, y_b)
            elif use_cahm:
                if is_adaptive:
                    d = loss_fn.compute_loss_dict(emb.float(), y)
                    per = d["losses"]
                else:
                    ef = emb.float()
                    ld = loss_fn.compute_loss(ef, y, None, ef, y)
                    per = ld["loss"]["losses"]
                w = _cahm_weights(y, cahm_d, cahm_alpha, device)
                loss = (per * w).mean()
            else:
                loss = loss_fn(emb, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
        scheduler.step()  # per-step schedule (warmup+cosine)

        total += loss.item() * y.size(0)
        seen += y.size(0)
    return total / max(seen, 1)


@torch.no_grad()
def validate(model, loss_fn, loader, device) -> Tuple[float, float]:
    """
    validation -> (val_loss, val_acc)
    - val_loss: ArcFace loss on the val set (logged; tiebreak for best-model selection)
    - val_acc : argmax over s*cos(theta) logits (true inference mode, no margin)
    """
    model.eval()
    embs, ys = [], []
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        edge = batch.get("edge_map")
        if edge is not None:
            edge = edge.to(device, non_blocking=True)
        embs.append(model(px, ln, edge).float().cpu())
        ys.append(batch["label"])
    E = torch.cat(embs).to(device)
    Y = torch.cat(ys).to(device)
    loss = loss_fn(E, Y).item()
    acc = (arcface_logits(loss_fn, E).argmax(dim=1) == Y).float().mean().item()
    return loss, acc

def save_history_plot(history: dict, out_png: str) -> None:
    """วาดกราฟ loss/accuracy — fail ได้ (headless) โดยไม่ทำ training ล้ม"""
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].plot(history["train_loss"], label="train")
        ax[0].plot(history["val_loss"], label="val")
        ax[0].set_title("ArcFace loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
        ax[1].plot(history["val_acc"], color="tab:green")
        ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=.3)
        fig.savefig(out_png, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"[log] บันทึกกราฟ history → {out_png}")
    except Exception as e:  # noqa: BLE001 — การวาดกราฟไม่ควรทำให้เทรนพัง
        print(f"(ข้ามการวาดกราฟ history: {e})")


# ============================================================ main training entry
def run_training(cfg: TrainConfig,
                 records_train: Optional[List[dict]] = None,
                 records_valid: Optional[List[dict]] = None,
                 tag: str = "") -> str:
    """
    Train once (single split) — returns path of the best checkpoint (highest val accuracy)

    checkpoint มี: model_state, arcface_state, classes, length_mean/std,
                   cfg (dict), epoch, val_loss, val_acc
    """
    os.makedirs(cfg.output_dir, exist_ok=True)
    seed_everything(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- ข้อมูล ----------
    if records_train is None or records_valid is None:
        records_train, records_valid, _ = resolve_records(cfg)
    all_recs = list(records_train) + list(records_valid)
    label2name = {r["label"]: r["class_name"] for r in all_recs}
    class_names = [label2name[i] for i in range(max(label2name) + 1)]  # index เรียงเสมอ

    # mean/std ของความยาว ← จาก train เท่านั้น (กัน leakage)
    length_stats = compute_length_stats(records_train, cfg.calibration_ratio)
    print(f"[data] train={len(records_train)} val={len(records_valid)} "
          f"classes={len(class_names)} length_mean={length_stats[0]:.2f} std={length_stats[1]:.2f}")

    flip_flags = build_flip_flags(class_names, cfg)
    use_sef = bool(getattr(cfg, "use_sef", False))
    ds_train = SurgicalInstrumentDataset(records_train, length_stats, cfg.img_size,
                                         cfg.calibration_ratio, flip_flags, training=True,
                                         bbox_margin=cfg.bbox_margin, use_sef=use_sef)
    ds_val = SurgicalInstrumentDataset(records_valid, length_stats, cfg.img_size,
                                       cfg.calibration_ratio, flip_flags=None, training=False,
                                       bbox_margin=cfg.bbox_margin, use_sef=use_sef)
    pin = device.type == "cuda"
    dl_train = DataLoader(ds_train, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=pin)
    dl_val = DataLoader(ds_val, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=pin)
    # ---------- โมเดล + loss ----------
    model = SurgicalDinoFusion(
        backbone_name=cfg.backbone_name, finetune_mode=cfg.finetune_mode,
        lora_r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        partial_last_blocks=cfg.partial_last_blocks, head_dropout=cfg.head_dropout,
        use_attention_pool=getattr(cfg, "use_attention_pool", True),
        use_sef=use_sef,
    ).to(device)
    print(f"[model] trainable params = {count_trainable(model):,} (mode={cfg.finetune_mode}, sef={use_sef})")

    # ArcFace — ปกติหรือ LGMS (per-class margin)
    if bool(getattr(cfg, "use_lgms", False)):
        class_means = compute_class_length_means(records_train, cfg.calibration_ratio)
        margins = compute_lgms_margins(class_means, len(class_names),
                                       m_base=cfg.margin, gamma=cfg.lgms_gamma, k=cfg.lgms_k)
        print(f"[LGMS] margins per class: {[f'{m:.1f}' for m in margins]}")
        loss_fn = AdaptiveArcFaceLoss(num_classes=len(class_names), embedding_size=model.embed_dim,
                                      margin_per_class=margins, scale=cfg.scale).to(device)
    else:
        # ArcFace: margin หน่วยองศา (~28.6° = 0.5 rad), s=64 — บังคับ embedding ให้
        # intra-class แน่น / inter-class ห่าง เหมาะกับ class ที่รูปทรงใกล้กันมาก
        loss_fn = ArcFaceLoss(num_classes=len(class_names), embedding_size=model.embed_dim,
                              margin=cfg.margin, scale=cfg.scale).to(device)
    if cfg.finetune_mode == "partial":
        lr_bb = cfg.lr_backbone
    elif cfg.finetune_mode == "lora":
        lr_bb = cfg.lr_lora
    else:
        lr_bb = None
    optimizer = AdamW(model.param_groups(cfg.lr_head, lr_bb), weight_decay=cfg.weight_decay)

    total_steps = max(1, len(dl_train)) * cfg.epochs
    warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))
    scheduler = LambdaLR(optimizer, lr_lambda=lambda s: warmup_cosine_factor(s, warmup_steps, total_steps))

    if device.type == "cuda":
        try:
            scaler = torch.amp.GradScaler("cuda")   # torch >= 2.3
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler()    # fallback torch เก่า
    else:
        scaler = None

    # ---------- loop + early stopping ----------
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    # เลือก best ด้วย "val_acc" (val_loss เป็น tiebreak) — ผ่านการจำลองพบว่าบนข้อมูลน้อย
    # ArcFace val_loss กับ val_acc แย่งกัน (loss ต่ำสุด ≠ โมเดลดีที่สุด); spec เดิมให้ดู val loss
    best = {"val_loss": float("inf"), "val_acc": -1.0, "epoch": -1, "model": None, "arcface": None}
    bad_epochs = 0
    cahm_d = None  # (C,C) EMA state สำหรับ CAHM
    use_cahm = bool(getattr(cfg, "use_cahm", False))
    cahm_start = int(getattr(cfg, "cahm_start_epoch", 10))
    cahm_beta = float(getattr(cfg, "cahm_beta", 0.9))

    for epoch in range(1, cfg.epochs + 1):
        # ส่ง cahm_d ให้ epoch นี้ถ้าถึงเวลาเริ่มแล้ว
        cur_cahm = cahm_d if (use_cahm and epoch > cahm_start and cahm_d is not None) else None
        tl = train_one_epoch(model, loss_fn, dl_train, optimizer, scheduler, scaler, device, cfg, cahm_d=cur_cahm)
        vl, va = validate(model, loss_fn, dl_val, device)
        history["train_loss"].append(tl); history["val_loss"].append(vl); history["val_acc"].append(va)

        # CAHM: อัปเดต difficulty หลัง validate (ใช้สำหรับ epoch ถัดไป)
        if use_cahm and epoch >= cahm_start:
            try:
                cm = _eval_confusion(model, loss_fn, dl_val, device, len(class_names))
                d_cur = _cahm_d_from_cm(cm)
                if cahm_d is None:
                    cahm_d = d_cur
                else:
                    cahm_d = cahm_beta * cahm_d + (1 - cahm_beta) * d_cur
                # log คู่สับสน top1 สำหรับดีบัก
                flat = [(i, j, cahm_d[i, j]) for i in range(len(class_names)) for j in range(len(class_names)) if i != j]
                flat.sort(key=lambda x: -x[2])
                if flat:
                    i, j, v = flat[0]
                    print(f"  [CAHM] top confused: {class_names[i]}↔{class_names[j]} d={v:.3f}")
            except Exception as e:
                print(f"  [CAHM] skip update: {e}")

        improved = va > best["val_acc"] + 1e-4 or \
            (va >= best["val_acc"] - 1e-4 and vl < best["val_loss"] - 1e-4)
        star = ""
        if improved:
            best.update(val_loss=vl, val_acc=va, epoch=epoch,
                        model={k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                        arcface={k: v.detach().cpu().clone() for k, v in loss_fn.state_dict().items()})
            bad_epochs = 0
            star = "  *best*"
            # เซฟทันที — หยุดกลางคันก็ไม่หาย (early-stop ก่อนก็มีไฟล์)
            try:
                ckpt_immediate = os.path.join(cfg.output_dir, f"best_model{tag}.pt")
                torch.save({
                    "model_state": best["model"],
                    "arcface_state": best["arcface"],
                    "classes": class_names,
                    "length_mean": float(length_stats[0]),
                    "length_std": float(length_stats[1]),
                    "calibration_ratio": cfg.calibration_ratio,
                    "cfg": cfg.to_dict(),
                    "epoch": best["epoch"],
                    "val_loss": float(best["val_loss"]),
                    "val_acc": float(best["val_acc"]),
                }, ckpt_immediate)
            except Exception as e:
                print(f"(ข้ามเซฟ immediate: {e})")
        else:
            bad_epochs += 1
        cur_lr = scheduler.get_last_lr()[0]
        print(f"{tag}[epoch {epoch:03d}/{cfg.epochs}] train={tl:.4f} val={vl:.4f} "
              f"val_acc={va:.4f} lr={cur_lr:.2e}{star}", flush=True)

        if bad_epochs >= cfg.patience:
            print(f"[early stop] ไม่ปรับปรุง {cfg.patience} epoch — หยุดที่ epoch {epoch}")
            break

    # ---------- save best checkpoint ----------
    ckpt_path = os.path.join(cfg.output_dir, f"best_model{tag}.pt")
    torch.save({
        "model_state": best["model"],
        "arcface_state": best["arcface"],
        "classes": class_names,
        "length_mean": float(length_stats[0]),
        "length_std": float(length_stats[1]),
        "calibration_ratio": cfg.calibration_ratio,
        "cfg": cfg.to_dict(),
        "epoch": best["epoch"],
        "val_loss": float(best["val_loss"]),
        "val_acc": float(best["val_acc"]),
    }, ckpt_path)
    save_history_plot(history, os.path.join(cfg.output_dir, f"history{tag}.png"))

    print(f"[done] best epoch={best['epoch']} val_loss={best['val_loss']:.4f} "
          f"val_acc={best['val_acc']:.4f} → {ckpt_path}")
    return ckpt_path


# ============================================================ k-fold CV
def run_kfold(cfg: TrainConfig, k: Optional[int] = None) -> Tuple[List[str], List[float]]:
    """
    Stratified k-fold CV บนข้อมูลทั้งหมด (train∪valid) — แม่นกว่า single split
    เมื่อข้อมูลมีแค่ ~30 ภาพ/class; คืน (paths, val_acc ต่อ fold)
    """
    from sklearn.model_selection import StratifiedKFold
    tr, va, _ = resolve_records(cfg)
    all_recs = tr + va
    y = [r["label"] for r in all_recs]
    skf = StratifiedKFold(n_splits=k or cfg.kfold, shuffle=True, random_state=cfg.seed)

    paths, accs = [], []
    for i, (idx_tr, idx_va) in enumerate(skf.split(all_recs, y)):
        rec_tr = [all_recs[j] for j in idx_tr]
        rec_va = [all_recs[j] for j in idx_va]
        print(f"\n========== Fold {i + 1}/{skf.n_splits} "
              f"(train={len(rec_tr)} val={len(rec_va)}) ==========")
        path = run_training(cfg, rec_tr, rec_va, tag=f"_fold{i + 1}")
        paths.append(path)
        ckpt = torch_load_compat(path)
        accs.append(float(ckpt["val_acc"]))

    print("\n===== K-FOLD SUMMARY =====")
    for i, a in enumerate(accs):
        print(f"fold {i + 1}: val_acc={a:.4f}")
    print(f"mean={np.mean(accs):.4f} ± {np.std(accs):.4f}")
    return paths, accs


# ============================================================ CLI
def main(argv=None) -> None:
    from dataclasses import replace
    ap = argparse.ArgumentParser(description="เทรน DINOv2+length-fusion+ArcFace จำแนกเครื่องมือผ่าตัด")
    ap.add_argument("--data_dir", default="dataset")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--batch_size", type=int, default=None)
    ap.add_argument("--img_size", type=int, default=None)
    ap.add_argument("--finetune_mode", choices=["lora", "partial", "frozen"], default=None)
    ap.add_argument("--kfold", type=int, default=None, help="เช่น 5 → Stratified 5-fold CV")
    ap.add_argument("--calibration_ratio", type=float, default=None,
                    help="cm/pixel จาก object อ้างอิง (ไม่ใส่ = ใช้ pixel)")
    ap.add_argument("--output_dir", default=None)
    ap.add_argument("--seed", type=int, default=None)
    # CAHM / LGMS / SEF — เปิด/ปิดอัลกอริทึมเสริม
    ap.add_argument("--use_cahm", action="store_true", help="เปิด CAHM (confusion-aware hard mining)")
    ap.add_argument("--use_lgms", action="store_true", help="เปิด LGMS (length-gated margin scaling)")
    ap.add_argument("--use_sef", action="store_true", help="เปิด SEF (Scharr edge fusion)")
    ap.add_argument("--cahm_alpha", type=float, default=None)
    ap.add_argument("--cahm_beta", type=float, default=None)
    ap.add_argument("--lgms_gamma", type=float, default=None)
    ap.add_argument("--lgms_k", type=int, default=None)
    args = ap.parse_args(argv)

    overrides = {}
    for k, v in vars(args).items():
        if k == "calibration_ratio":
            continue
        if v is None:
            continue
        # store_true flags: False หมายถึงไม่ใส่ → ไม่ override (คงค่าเดิม False)
        if isinstance(v, bool) and not v:
            continue
        overrides[k] = v
    if args.calibration_ratio is not None:
        overrides["calibration_ratio"] = args.calibration_ratio
    cfg = replace(TrainConfig(), **overrides)
    if cfg.kfold and cfg.kfold > 1:
        run_kfold(cfg)
    else:
        path = run_training(cfg)
        print("checkpoint:", path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
# -*- coding: utf-8 -*-
"""
evaluate.py — ประเมิน checkpoint ที่เทรนแล้ว

ใช้จาก notebook/สคริปต์:
    from evaluate import evaluate_checkpoint
    metrics = evaluate_checkpoint("outputs/best_model.pt")

ได้ทั้ง accuracy รวม, classification report ราย class, confusion matrix
(heatmap) และ "คู่ class ที่สับสนบ่อยสุด" ซึ่งมักเป็นคู่ที่ต่างกันแค่ขนาด
"""
import os
from dataclasses import replace
from typing import List, Optional

import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                             confusion_matrix)

from config import TrainConfig
from dataset import SurgicalInstrumentDataset
from model import SurgicalDinoFusion, arcface_logits
from pytorch_metric_learning.losses import ArcFaceLoss
from train import resolve_records, torch_load_compat



def load_bundle(ckpt_path: str, device: Optional[torch.device] = None) -> dict:
    """
    โหลด checkpoint → สร้าง model + ArcFace head ให้พร้อม inference
    (cfg ถูกเก็บไว้ใน checkpoint เวลาเทรน → reproduce โครงสร้างได้เป๊ะ)
    """
    ckpt = torch_load_compat(ckpt_path)
    cfg = TrainConfig(**ckpt["cfg"])
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ต้องสร้างด้วย finetune_mode เดิม เพื่อให้ชื่อ key ของ state_dict ตรงกัน
    model = SurgicalDinoFusion(
        backbone_name=cfg.backbone_name, finetune_mode=cfg.finetune_mode,
        lora_r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        partial_last_blocks=cfg.partial_last_blocks, head_dropout=cfg.head_dropout,
        use_attention_pool=getattr(cfg, "use_attention_pool", True),
        use_sef=bool(getattr(cfg, "use_sef", False)),
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(device).eval()

    classes: List[str] = ckpt["classes"]
    arcface = ArcFaceLoss(num_classes=len(classes), embedding_size=model.embed_dim,
                          margin=cfg.margin, scale=cfg.scale)
    try:
        arcface.load_state_dict(ckpt["arcface_state"])
    except Exception:
        # ถ้าเทรนด้วย LGMS (AdaptiveArcFace) แต่ประเมินด้วย ArcFace ปกติ — W ยังโหลดได้
        # ลองโหลดแบบ strict=False
        arcface.load_state_dict(ckpt["arcface_state"], strict=False)
    arcface.to(device)

    return {"model": model, "arcface": arcface, "classes": classes, "cfg": cfg,
            "device": device,
            "length_mean": float(ckpt["length_mean"]), "length_std": float(ckpt["length_std"]),
            "calibration_ratio": ckpt.get("calibration_ratio")}

@torch.no_grad()
def predict_all(bundle: dict, records: List[dict]):
    """รันทั้ง validation set → (y_true, y_pred, confidence ของ class ที่ทำนาย)"""
    cfg = bundle["cfg"]
    device = bundle["device"]
    length_stats = (bundle["length_mean"], bundle["length_std"])
    use_sef = bool(getattr(cfg, "use_sef", False))
    ds = SurgicalInstrumentDataset(records, length_stats, cfg.img_size,
                                   cfg.calibration_ratio, flip_flags=None, training=False,
                                   bbox_margin=getattr(cfg, "bbox_margin", 0.0),
                                   use_sef=use_sef)
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    y_true, y_pred, y_conf = [], [], []
    for batch in dl:
        px = batch["image"].to(device)
        ln = batch["length"].to(device)
        edge = batch.get("edge_map")
        if edge is not None:
            edge = edge.to(device)
        emb = bundle["model"](px, ln, edge)
        logits = arcface_logits(bundle["arcface"], emb.float())
        probs = torch.softmax(logits, dim=-1)
        conf, pred = probs.max(dim=-1)
        y_pred += pred.cpu().tolist()
        y_conf += conf.cpu().tolist()
        y_true += batch["label"].tolist()
    return np.array(y_true), np.array(y_pred), np.array(y_conf)


def plot_confusion_matrix(cm: np.ndarray, class_names: List[str],
                          save_path: Optional[str] = None, figsize=(12, 10)):
    """heatmap ของ confusion matrix (seaborn ถ้ามี, ไม่มีก็ matplotlib ล้วน)"""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=figsize)
    try:
        import seaborn as sns
        sns.heatmap(cm, annot=True, fmt="d", cmap="viridis",
                    xticklabels=class_names, yticklabels=class_names, ax=ax)
    except ImportError:
        im = ax.imshow(cm, cmap="viridis")
        fig.colorbar(im, ax=ax)
        ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=90)
        ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches="tight")
        print(f"[log] บันทึก confusion matrix → {save_path}")
    return fig


def print_top_confused(cm: np.ndarray, class_names: List[str], top_n: int = 10) -> None:
    """พิมพ์คู่ class ที่สับสนบ่อยสุด (จริง → ทำนาย) — จุดที่ต้องแก้ต่อ เช่น เพิ่มฟีเจอร์ขนาด"""
    pairs = [(int(cm[i, j]), i, j)
             for i in range(len(class_names)) for j in range(len(class_names))
             if i != j and cm[i, j] > 0]
    if not pairs:
        print("ไม่มีความสับสนข้าม class เลย 🎉")
        return
    pairs.sort(reverse=True)
    print("\nคู่ class ที่สับสนบ่อยสุด (จริง → ทำนาย):")
    for cnt, i, j in pairs[:top_n]:
        print(f"  {class_names[i]} → {class_names[j]} : {cnt} ครั้ง")


def evaluate_checkpoint(ckpt_path: str, data_dir: Optional[str] = None,
                        show_plot: bool = True, save_dir: Optional[str] = None) -> dict:
    """
    ประเมิน checkpoint บน validation set → dict ที่มี
      accuracy / balanced_accuracy / report / confusion_matrix / cm_path / fig
    """
    bundle = load_bundle(ckpt_path)
    cfg = bundle["cfg"]
    if data_dir:
        cfg = replace(cfg, data_dir=data_dir)
    _, va_records, _ = resolve_records(cfg)
    if len(va_records) == 0:
        raise ValueError("validation set ว่าง — เช็ค data_dir/val_fraction")

    y_true, y_pred, _ = predict_all(bundle, va_records)
    classes = bundle["classes"]
    acc = float((y_true == y_pred).mean())
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))

    present = sorted(set(y_true.tolist()))
    names_present = [classes[i] for i in present]
    print(f"\n===== Evaluation ({len(va_records)} samples) =====")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print("\n" + classification_report(
        [classes[i] for i in y_true], [classes[i] for i in y_pred],
        labels=names_present, digits=3, zero_division=0))

    out_dir = save_dir or cfg.output_dir
    os.makedirs(out_dir, exist_ok=True)
    cm_path = os.path.join(out_dir, "confusion_matrix.png")
    fig = plot_confusion_matrix(cm, classes, save_path=cm_path)
    if show_plot:
        try:
            import matplotlib.pyplot as plt
            plt.show()
        except Exception:
            pass
    print_top_confused(cm, classes)

    return {"accuracy": acc, "balanced_accuracy": bal_acc,
            "report": classification_report(
                [classes[i] for i in y_true], [classes[i] for i in y_pred],
                output_dict=True, zero_division=0),
            "confusion_matrix": cm, "cm_path": cm_path, "fig": fig}


In [ ]:
%%writefile infer.py
# -*- coding: utf-8 -*-
"""
infer.py — inference ภาพใหม่ 1 ภาพ (+mask) → class + confidence

ใช้จาก notebook/สคริปต์:
    from infer import load_pipeline, predict_record, predict_file
    pack = load_pipeline("outputs/best_model.pt")
    res  = predict_file(pack, image_path="x.jpg", mask_png="x_mask.png")
    # หรือมี COCO json: predict_file(pack, "x.jpg", coco_json="_annotations.coco.json",
    #                                 image_filename="x.jpg")

CLI:
    python infer.py --ckpt outputs/best_model.pt --image x.jpg --mask_png x_mask.png

หมายเหตุ: confidence คือ softmax ของ s·cos(θ) — ใช้เปรียบเทียบ "ความมั่นใจสัมพัทธ์"
ระหว่าง class ได้ แต่ไม่ใช่ probability ที่ calibrated จริง
"""
import argparse
import json
import os
from typing import List, Optional

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from dataset import build_tensor_transform, mask_from_coco_segmentation, measure_length_px, scharr_edge_map
from model import arcface_logits


def load_pipeline(ckpt_path: str, device: Optional[torch.device] = None) -> dict:
    """โหลด checkpoint → pack (model, arcface head, transform, metadata)"""
    from evaluate import load_bundle
    pack = load_bundle(ckpt_path, device)
    pack["tensor_tf"] = build_tensor_transform(pack["cfg"].img_size)
    # เก็บ cfg เป็น dict เพื่อให้ predict_array อ่าน use_tta/use_sef ได้
    if hasattr(pack["cfg"], "to_dict"):
        # เก็บ object ไว้ด้วยสำหรับ img_size
        pack["_cfg_obj"] = pack["cfg"]
        pack["cfg"] = pack["cfg"].to_dict()
    return pack


def _predict_once(pack: dict, image_rgb: np.ndarray, ln_norm: float) -> torch.Tensor:
    """ทำนายครั้งเดียว → คืน logits tensor (num_classes,)"""
    tensor = pack["tensor_tf"](image=image_rgb)["image"][None].to(pack["device"])
    ln = torch.tensor([ln_norm], dtype=torch.float32, device=pack["device"])
    edge = None
    cfg = pack.get("cfg", {})
    if cfg.get("use_sef", False):
        # Scharr edge map จากภาพหลัง crop/resize เดียวกับที่ใช้ train
        gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
        edge_np = scharr_edge_map(gray)
        h, w = image_rgb.shape[:2]
        # edge ต้อง resize ให้ตรงกับ tensor (img_size)
        img_size = cfg.get("img_size", 616)
        edge_resized = cv2.resize(edge_np, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        edge = torch.from_numpy(edge_resized).unsqueeze(0).unsqueeze(0).to(pack["device"]).float()
    emb = pack["model"](tensor, ln, edge)
    return arcface_logits(pack["arcface"], emb.float())[0]

@torch.no_grad()
def predict_array(pack: dict, image_rgb: np.ndarray,
                  mask_gray: Optional[np.ndarray] = None) -> dict:
    """
    ทำนาย 1 ภาพ พร้อม TTA (Test-Time Augmentation)
      image_rgb : uint8 (H,W,3) RGB
      mask_gray : binary mask ของเครื่องมือ (H,W) — ถ้าไม่มี จะใช้ค่าความยาวเฉลี่ย
                  ของ train แทน (โมเดลยังทำนายได้ แต่แม่นยำน้อยลงกับคู่ class ต่างขนาด)

    TTA: original + horizontal flip → average logits → เสถียรกว่าทำนายครั้งเดียว
    """
    ratio = pack.get("calibration_ratio")
    if mask_gray is not None:
        length = measure_length_px(mask_gray) * (ratio if ratio else 1.0)
    else:
        length = pack["length_mean"]  # neutral fallback

    ln = (length - pack["length_mean"]) / pack["length_std"]

    # TTA: original + horizontal flip → average logits
    use_tta = pack.get("cfg", {}).get("use_tta", False) if isinstance(pack.get("cfg"), dict) else False
    if use_tta:
        logits_orig = _predict_once(pack, image_rgb, ln)
        # horizontal flip ของ mask ด้วย (ถ้ามี) ไม่ต้อง flip ความยาว — ความยาวไม่เปลี่ยน
        img_flip = np.ascontiguousarray(image_rgb[:, ::-1, :])
        logits_flip = _predict_once(pack, img_flip, ln)
        logits = (logits_orig + logits_flip) / 2.0
    else:
        logits = _predict_once(pack, image_rgb, ln)

    probs = F.softmax(logits, dim=-1).cpu()

    top3 = probs.topk(3)
    classes: List[str] = pack["classes"]
    return {
        "class": classes[int(top3.indices[0])],
        "confidence": float(top3.values[0]),
        "top3": [{"class": classes[int(i)], "prob": float(p)}
                 for p, i in zip(top3.values, top3.indices)],
        "length_used": float(length),
        "tta_used": use_tta,
    }


def predict_record(pack: dict, record: dict) -> dict:
    """ทำนายจาก record ที่ได้จาก load_coco_records() (ใช้ segmentation polygon ใน record)"""
    bgr = cv2.imread(record["image_path"], cv2.IMREAD_COLOR)
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    mask = mask_from_coco_segmentation(record["segmentation"], record["height"], record["width"])
    res = predict_array(pack, img, mask)
    res["truth"] = record["class_name"]
    return res


def predict_file(pack: dict, image_path: str, mask_png: Optional[str] = None,
                 coco_json: Optional[str] = None,
                 image_filename: Optional[str] = None) -> dict:
    """
    ทำนายจากไฟล์:
      - mask_png   : ไฟล์ mask (ขาว/ดำ) ถ้ามี
      - coco_json  : หรือชี้ไฟล์ annotation แล้วระบุ image_filename → ใช้ polygon ann แรกของรูปนั้น
    """
    bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise IOError(f"อ่านภาพไม่สำเร็จ: {image_path}")
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    mask = None
    if mask_png:
        m = cv2.imread(mask_png, cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise IOError(f"อ่าน mask ไม่สำเร็จ: {mask_png}")
        mask = ((m > 127).astype(np.uint8)) * 255
    elif coco_json:
        fname = image_filename or os.path.basename(image_path)
        with open(coco_json, "r", encoding="utf-8") as f:
            coco = json.load(f)
        images = {im["id"]: im for im in coco["images"]}
        target = next((im for im in coco["images"] if im["file_name"] == fname), None)
        if target is None:
            raise KeyError(f"ไม่พบ {fname} ใน {coco_json}")
        ann = next((a for a in coco["annotations"] if a["image_id"] == target["id"]), None)
        if ann is None:
            raise KeyError(f"{fname} ไม่มี annotation")
        mask = mask_from_coco_segmentation(ann["segmentation"],
                                           int(images[target["id"]]["height"]),
                                           int(images[target["id"]]["width"]))
    return predict_array(pack, img, mask)


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description="Inference DINOv2+ArcFace")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--image", required=True)
    ap.add_argument("--mask_png", default=None)
    ap.add_argument("--coco_json", default=None)
    args = ap.parse_args(argv)

    pack = load_pipeline(args.ckpt)
    res = predict_file(pack, args.image, mask_png=args.mask_png, coco_json=args.coco_json)
    print(json.dumps(res, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 3) Sanity check ข้อมูล

เช็ค: จำนวนภาพต่อ class, mask overlay ถูกไหม, ความยาวที่วัดได้ sensible ไหม — **ก่อนเทรนเสมอ**

In [ ]:
CALIB_RATIO = None   # ← cm/pixel ถ้ามี object อ้างอิง (เช่น 0.05) ; None = ใช้ pixel

import sys; sys.path.insert(0, "/content")
from collections import Counter

from dataset import load_coco_records, visualize_records, compute_length_stats

train_recs, class_names = load_coco_records(DATA_DIR, "train")
dist = Counter(r["class_name"] for r in train_recs)
print(f"class ทั้งหมด: {len(class_names)}")
for n in class_names:
    print(f"  {n:30s}: {dist[n]} ภาพ")

stats = compute_length_stats(train_recs, CALIB_RATIO)
unit = "cm" if CALIB_RATIO else "px"
print(f"\nlength stats (train): mean={stats[0]:.1f} {unit}, std={stats[1]:.1f}")

fig = visualize_records(train_recs, calibration_ratio=CALIB_RATIO, n=6, seed=7)

## 3.5) เช็คว่า weight ของ DINOv2 พร้อมหรือยัง (frozen kNN probe — ไม่ต้องเทรน)

ใช้ DINOv2 **frozen** ดึง feature → ทำนายด้วย 1-nearest-neighbor:
- accuracy สูงกว่า random (1/14 ≈ 7%) **มากๆ** (เช่น >40-50%) = feature แยก class ได้อยู่แล้ว เทรนต่อคุ้ม
- accuracy ต่ำเฉียด random = domain gap แรงเกิน → ลอง `img_size=518` (คมขึ้น) หรือ backbone ใหญ่ขึ้น

In [ ]:
# ponytail: probe ไม่เทรนสัก step — เช็คคุณภาพ feature ก่อนลงทุนเทรนจริง
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from config import TrainConfig
from dataset import SurgicalInstrumentDataset, compute_length_stats
from model import SurgicalDinoFusion
from train import resolve_records

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
probe_model = SurgicalDinoFusion(finetune_mode="frozen").to(device).eval()

cfg0 = TrainConfig(data_dir=DATA_DIR)
tr_recs, va_recs, probe_classes = resolve_records(cfg0)
probe_stats = compute_length_stats(tr_recs, CALIB_RATIO)
flip_all = [True] * len(probe_classes)

@torch.no_grad()
def embed(recs, training):
    ds = SurgicalInstrumentDataset(recs, probe_stats, cfg0.img_size, CALIB_RATIO,
                                   flip_all if training else None, training)
    E, Y = [], []
    for b in DataLoader(ds, batch_size=32, num_workers=2):
        out = probe_model.backbone(pixel_values=b["image"].to(device)).last_hidden_state
        E.append(out[:, 0].cpu())          # CLS ดิบ — ไม่ผ่าน fusion head (head ยังสุ่มอยู่)
        Y.append(b["label"])
    return torch.cat(E), torch.cat(Y)

Etr, ytr = embed(tr_recs, True)            # train-side aug เปิดได้ = ฟรี data augmentation ของ probe
Eva, yva = embed(va_recs, False)
sim = F.normalize(Eva, dim=1) @ F.normalize(Etr, dim=1).T
pred = ytr[sim.argmax(dim=1)]
acc = (pred == yva).float().mean().item()
print(f"kNN probe accuracy: {acc:.3f}   (random = {1 / len(probe_classes):.3f})")
if acc < 0.30:
    print("⚠️ feature แยก class ค่อนข้างไม่ได้ — ลอง img_size=518 หรือ dinov2-base ก่อนเทรนจริง")
else:
    print("✅ feature ใช้ได้ — ไปเทรนต่อได้")

## 4) เทรน

- `finetune_mode="lora"` → เทรนเฉพาะ LoRA adapter (~พารามิเตอร์น้อยมาก) กัน overfitting
- early stopping จาก validation loss, checkpoint ที่ดีที่สุดถูกเซฟอัตโนมัติ
- อยากได้ผลประเมินแบบ cross-validation → ตั้ง `kfold=5`

In [ ]:
from config import TrainConfig
from train import run_training, run_kfold

cfg = TrainConfig(
    data_dir=DATA_DIR,
    img_size=616,            # 616=44×14 — เต็มที่ batch 32 บน T4
    batch_size=32,
    finetune_mode="lora",     # "lora" (แนะนำ) | "partial" | "frozen"
    epochs=50,
    num_workers=2,
    # ---- ตัวเลือกเสริม ----
    # calibration_ratio=CALIB_RATIO,
    # flip_allowed=["class_a", "class_b"],  # เฉพาะ class ที่ flip ได้ (ที่เหลือห้าม flip)
    # kfold=5,                              # Stratified 5-fold CV
)

if cfg.kfold:
    paths, accs = run_kfold(cfg)
    best_ckpt = paths[accs.index(max(accs))]   # เลือก fold ที่ดีที่สุด
else:
    best_ckpt = run_training(cfg)

print("\ncheckpoint ที่ดีที่สุด:", best_ckpt)

## 5) ประเมินผล — accuracy + confusion matrix

ดู heatmap ช่อง off-diagonal สีสว่าง = คู่ class ที่โมเดลสับสน (มักเป็นคู่ต่างกันแค่ขนาด)

In [ ]:
from evaluate import evaluate_checkpoint

metrics = evaluate_checkpoint(best_ckpt)
print(f"\nAccuracy: {metrics['accuracy']:.4f}")

## 5.5) Phase 3 — เทรน ablation 6 สูตร (baseline vs CAHM/LGMS/SEF)

รันทีละสูตรแล้วเก็บ checkpoint — **ทำบน Colab แบบนี้** เพื่อวัดผลเปรียบเทียบในเซลล์ถัดไป
แต่ละสูตร early-stop ที่ ~20-35 epoch (patience 12) | 616px batch32 บน T4 ~9-10GB — ถ้า OOM จะลด batch เหลือ 16 อัตโนมัติ


In [ ]:
from config import TrainConfig
from train import run_training
import torch

ablation = {
    "baseline":  {},
    "cahm":      {"use_cahm": True},
    "lgms":      {"use_lgms": True},
    "sef":       {"use_sef": True},
    "cahm_lgms": {"use_cahm": True, "use_lgms": True},
    "all":       {"use_cahm": True, "use_lgms": True, "use_sef": True},
}

ckpt_map = {}
for name, extra in ablation.items():
    print(f"\n{'='*20} {name} {extra} {'='*20}")
    cfg_ab = TrainConfig(
        data_dir=DATA_DIR,
        img_size=616, batch_size=32,
        finetune_mode="lora", epochs=50, num_workers=2,
        output_dir=f"outputs_ablation/{name}",
        **extra
    )
    try:
        ckpt = run_training(cfg_ab)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("[OOM] ลด batch 32 → 16 แล้วลองใหม่")
            torch.cuda.empty_cache()
            cfg_ab.batch_size = 16
            ckpt = run_training(cfg_ab)
        else:
            raise
    ckpt_map[name] = ckpt
    print(f"[{name}] ✅ {ckpt}")

print("\n--- ckpt_map ---")
for k,v in ckpt_map.items():
    print(f'{k}: "{v}"')

## 5.6) เปรียบเทียบ ablation — ตาราง Phase 3

วัด Val Acc / Balanced Acc / Needle_Holder↔Artery_Forceps error พร้อมกัน 6 สูตร
ผ่านไฟล์ `tools/evaluate_ablation.py` (รันได้ทั้งใน notebook และ CLI)


In [ ]:
from tools.evaluate_ablation import compare_checkpoints

# ใช้ ckpt_map จากเซลล์ก่อนหน้า ถ้ายังไม่ได้รัน ablation ให้ใส่ path เองได้:
# ckpt_map = {
#     "baseline": "outputs_ablation/baseline/best_model.pt",
#     "cahm": "outputs_ablation/cahm/best_model.pt",
#     ...
# }
results = compare_checkpoints(ckpt_map, data_dir=DATA_DIR, save_csv="outputs_ablation/ablation_results.csv")
print("\n--- สรุป --")
for r in results:
    if r.get("found"):
        print(f'{r["name"]:12s} acc={r["accuracy"]:.4f} bal={r["balanced_acc"]:.4f} {r["needle_detail"]}')

## 6) Inference ทดลอง — ภาพใหม่ + mask → class + confidence

In [ ]:
from infer import load_pipeline, predict_record
from train import resolve_records

pack = load_pipeline(best_ckpt)
_, valid_recs, _ = resolve_records(cfg)   # เอาภาพ val มาลอง 3 ตัว

for r in valid_recs[:3]:
    res = predict_record(pack, r)
    ok = "✅" if res["class"] == res["truth"] else "❌"
    tops = ", ".join(f'{t["class"]}:{t["prob"]:.2f}' for t in res["top3"])
    print(f'{ok} truth={res["truth"]:20s} pred={res["class"]:20s} conf={res["confidence"]:.3f}')
    print("      top3:", tops)

## 7) บันทึกโมเดลกลับ Drive / ดาวน์โหลด

In [ ]:
# ▸ ดาวน์โหลดกลับเครื่อง
from google.colab import files
files.download(best_ckpt)

# ▸ หรือคัดลอกไป Drive (mount ก่อนถ้ายังไม่ได้ mount)
# !cp "{best_ckpt}" /content/drive/MyDrive/

---
### 💡 Tips
- **calibration_ratio**: วาง object ที่รู้ความยาวจริง L (cm) ใต้กล้อง rig เดิม → `ratio = L / measure_length_px(mask)` — feature ความยาวจะมีหน่วย cm ตีความง่าย
- **handedness**: class ของซ้าย/ขวามือ → ใส่เฉพาะชื่อ class ที่ flip ได้ใน `flip_allowed`

### 🎯 Playbook คู่ class ที่แทบแยกไม่ออก (เช่น Universal forceps 150 vs 151, Root elevator งอ vs Root-tip elevator ตรง)
1. **เพิ่ม resolution ก่อนเป็นอย่างแรก** — head ของ forceps ที่ 224px เหลือ ~20px ตั้ง `img_size=518` (=37×14, DINOv2 รับได้) patch token จะพกรายละเอียด "หัว" เยอะขึ้น ~2.3×; GPU ไม่พอลด batch เป็น 8
2. **คู่ต่างความยาว** (elevator งอสั้น vs straight ยาว) — length feature จัดการอยู่แล้ว แต่ mask ต้องครอบปลายจริง (mask โดนเงา = ความยาวเพี้ยน = ฟีเจอร์เพี้ยน)
3. **ถ่ายข้อมูลเพิ่มแบบเจาะจง**: ดู confusion matrix ในขั้น 5 → ถ่ายเพิ่มเฉพาะคู่ที่สับสน โดยหมุนมุม yaw ทีละ ~30° (บางมุมหัวซ่อน = ต้องมีภาพมุมที่เห็นจุดต่าง)
4. **ยังสับสนหนัก?** แผน B: สองขั้น — แยกกลุ่มหยาบ (forceps/elevator/probe…) ก่อน แล้ว classifier เฉพาะคู่บน crop บริเวณหัว

### 📋 เช็คลิสต์ตอน annotate รอบใหม่ (Roboflow COCO Segmentation)
- polygon ครอบ **ทั้งชิ้น** รวมหัว-ปลาย แต่ **ไม่เอาเงา** (เงาทำ minAreaRect ยืวขึ้น)
- 1 annotation ต่อ 1 ชิ้นเครื่องมือ (1 ภาพหลายชิ้นได้ — pipeline crop รายชิ้นด้วย bbox_margin)
- ชื่อ class ตั้งแล้วอย่าเปลี่ยนทีหลัง (label = ลำดับการ sort ชื่อ)
- จำนวนต่อ class ห้ามหลุด ~25+ ภาพ และกระจายมุมถ่ายให้ทั่ว
- พื้นเขียว: pipeline มี shadow augmentation ให้แล้ว แต่ถ่ายให้แสงสม่ำเสมอที่ทำได้ยังดีที่สุด

- GPU: Runtime → Change runtime type → T4 GPU (CPU ได้แต่ช้า); `img_size=518` + T4 + batch 8 ≈ ไหว
